# BUSADMIN O712 — Assignment 1: Finding the Best Delivery Promise for Rosa's Pizza

**Links**
- GitHub repository for the Streamlit app: `<paste your GitHub repo URL here>`
- Deployed app on Streamlit Community Cloud: `<paste your app URL here>`

**Roadmap**
- **Part I (Describe):** How often are orders late, how long do deliveries take, and which of these two measures should drive Rosa's decisions?
- **Part II (Prescribe):** What does a late order cost, and which promise maximises net profit in each zone and time block?
- **Part III:** A Streamlit web app built on the functions in this notebook. The code lives in the GitHub repo linked above.
- **Appendix:** How AI was used, with the key prompts.

Throughout, **an order is late if its delivery time is strictly greater than the promised time** (arriving at exactly 45:00 on a 45-minute promise still keeps the promise).

## Setup
Colab starts a fresh machine every session, so this cell must be run each time the notebook is opened.

In [ ]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

# delivery_times is a simulator. A fixed seed makes every run of this notebook reproducible.
# Using the SAME seed for every promise we try also means that differences in profit between
# two promises come from the promise itself, not from random noise ("common random numbers").
SEED = 1

print("Zones:      ", ZONES)
print("Time blocks:", TIME_BLOCKS)
print("Costs:      ", COSTS)
print("Current promise:", PROMISE, "minutes")

times = delivery_times("Far West", "Fri/Sat eve", 45, seed=SEED)
print(len(times), "orders; average delivery time", round(times.mean(), 1), "minutes")

### Helper functions
Both Part I functions share the same logic for handling `'all'`. They expand `'all'` into the full list of zones and/or time blocks. They then **pool the individual orders** from every selected zone–time-block pair before computing the statistic.

Pooling matters. A pair with 230 orders should count more than a pair with 90 orders. If we took a simple average of the 12 pair-level percentages, the small, quiet segments would be over-weighted. Pooling weights each segment by its actual order volume, so the result is the true share of all of Rosa's orders.

In [ ]:
def _expand(zone, time_block):
    # Turn a zone / time-block request (which may be 'all') into a list of (zone, time_block) pairs.
    if zone == "all":
        zones = list(ZONES)
    elif zone in ZONES:
        zones = [zone]
    else:
        raise ValueError(f"zone must be 'all' or one of {ZONES}")

    if time_block == "all":
        blocks = list(TIME_BLOCKS)
    elif time_block in TIME_BLOCKS:
        blocks = [time_block]
    else:
        raise ValueError(f"time_block must be 'all' or one of {TIME_BLOCKS}")

    return [(z, b) for z in zones for b in blocks]


def pooled_times(zone, time_block, promise=PROMISE, seed=SEED):
    # All individual delivery times for the requested zone(s) and time block(s), in one array.
    arrays = [delivery_times(z, b, promise, seed=seed) for z, b in _expand(zone, time_block)]
    return np.concatenate(arrays)

---
# Part I. Describe: How bad is it, really?

## I(a) Percentage of late orders

In [ ]:
def late_rate(zone, time_block, promise=PROMISE, seed=SEED):
    # Percentage of orders delivered after the promised time.
    # zone and time_block can each be a specific name or 'all'.
    times = pooled_times(zone, time_block, promise, seed)
    if len(times) == 0:
        return float("nan")
    return 100 * np.mean(times > promise)


examples = [
    ("Far West", "Fri/Sat eve"),   # specific zone, specific time block
    ("all", "Lunch"),              # all zones, one time block
    ("Far West", "all"),           # one zone, all time blocks
    ("all", "all"),                # overall
]
for z, b in examples:
    print(f"Late rate  | zone = {z!r:12} time block = {b!r:15} -> {late_rate(z, b, 45):5.1f}%")

## I(b) Ranking zone–time-block pairs by late rate (45-minute promise)

In [ ]:
def rank_pairs(metric, promise=PROMISE, seed=SEED):
    # Compute metric(zone, time_block, promise, seed) for every pair and sort from highest to lowest.
    rows = []
    for z in ZONES:
        for b in TIME_BLOCKS:
            n = len(delivery_times(z, b, promise, seed=seed))
            rows.append((z, b, metric(z, b, promise, seed), n))
    return sorted(rows, key=lambda r: r[2], reverse=True)


late_ranking = rank_pairs(late_rate, 45)

print(f"{'Rank':<5}{'Zone':<12}{'Time block':<15}{'Late %':>8}{'Orders':>8}")
for i, (z, b, v, n) in enumerate(late_ranking, 1):
    print(f"{i:<5}{z:<12}{b:<15}{v:>7.1f}%{n:>8}")

## I(c) Average delivery time

In [ ]:
def avg_delivery_time(zone, time_block, promise=PROMISE, seed=SEED):
    # Average delivery time in minutes (pooled over orders). zone / time_block may be 'all'.
    times = pooled_times(zone, time_block, promise, seed)
    if len(times) == 0:
        return float("nan")
    return times.mean()


for z, b in examples:
    print(f"Avg time   | zone = {z!r:12} time block = {b!r:15} -> {avg_delivery_time(z, b, 45):5.1f} min")

## I(d) Ranking zone–time-block pairs by average delivery time (45-minute promise)

In [ ]:
avg_ranking = rank_pairs(avg_delivery_time, 45)

print(f"{'Rank':<5}{'Zone':<12}{'Time block':<15}{'Avg min':>8}{'Orders':>8}")
for i, (z, b, v, n) in enumerate(avg_ranking, 1):
    print(f"{i:<5}{z:<12}{b:<15}{v:>8.1f}{n:>8}")

## I(e) Which measure is more relevant to Rosa's decisions?

The cell below puts the two rankings side by side. It also shows each pair's 90th-percentile delivery time and flags segments where **the average is inside the promise but a meaningful share of orders is still late.**

In [ ]:
late_rank = {(z, b): i for i, (z, b, _, _) in enumerate(late_ranking, 1)}
avg_rank  = {(z, b): i for i, (z, b, _, _) in enumerate(avg_ranking, 1)}

print(f"{'Zone':<12}{'Time block':<15}{'Late %':>8}{'Avg':>7}{'P90':>7}{'Rank(b)':>9}{'Rank(d)':>9}")
hidden = []
for z, b, late, n in late_ranking:
    t = delivery_times(z, b, 45, seed=SEED)
    avg, p90 = t.mean(), np.percentile(t, 90)
    print(f"{z:<12}{b:<15}{late:>7.1f}%{avg:>7.1f}{p90:>7.1f}{late_rank[(z, b)]:>9}{avg_rank[(z, b)]:>9}")
    if avg <= 45 and late >= 10:
        hidden.append((z, b, avg, late))

moved = sum(late_rank[k] != avg_rank[k] for k in late_rank)
print(f"\n{moved} of {len(late_rank)} pairs are ranked differently by the two measures.")
print("Pairs whose AVERAGE is within 45 min but still have >=10% late orders:")
for z, b, avg, late in hidden:
    print(f"  {z} / {b}: average {avg:.1f} min, yet {late:.1f}% of orders are late")
if not hidden:
    print("  (none at this threshold)")

**The late rate (b) is the more relevant measure for Rosa's decisions.** There are four reasons.

1. **Rosa's costs are triggered by lateness, not by the average.** The refund is paid, and the customer's future orders are lost, only when an order misses the promise. An order that arrives at 44 minutes costs Rosa nothing, while one at 46 minutes costs the full refund plus churn. The late rate counts exactly the events that cost money. The average does not.

2. **The average hides the tail.** A segment can have an average comfortably below 45 minutes and still break the promise on a substantial share of orders, because a few very slow deliveries pull the tail past the threshold. The table above shows this directly: compare the average with the P90 column, and see the pairs flagged as having an average within 45 minutes but a high late rate. Conversely, two segments with similar averages can have quite different late rates if one is more variable. An average-based ranking would point Rosa's attention to the wrong places.

3. **The decision variable is a threshold.** Rosa's lever is the promise itself, a cut-off. The question "if I promise *P* minutes, what share of orders will exceed *P*?" is answered by the late rate, which feeds directly into the profit calculation in Part II. The average cannot be converted into a refund bill.

4. **It is what customers experience.** A customer judges Rosa against the promise on their own order, not against the average across all orders.

The average delivery time is still useful as **diagnostic context**. It tells us where operations are simply slower (for example, distance to the Far West, or congestion on weekend evenings) and roughly where a realistic promise might sit. But for choosing a promise and valuing its cost, **the late rate is the measure to manage to.** The comparison above also shows that the two rankings do not always agree. Where they disagree, the late-rate ranking identifies the segments that actually cost Rosa money.

---
# Part II. Exploring the best promise

## II(a) Total cost per late order
A late order costs Rosa in two ways:
- **Refund:** a direct cash cost per late order.
- **Churn:** the customer places fewer future orders. Each lost order is a lost profit margin.

$$\text{cost per late order} = \text{refund} + \text{churn (lost orders)} \times \text{profit margin per order}$$

The helper `unpack_costs` finds the three figures in `COSTS` by name. The same functions therefore work with the `COSTS` dictionary and with the values a user types into the web app.

In [ ]:
def unpack_costs(costs):
    # Return (refund, churn, margin) from a costs dictionary, matching keys by name.
    def find(*words):
        for key, value in costs.items():
            if any(w in key.lower() for w in words):
                return float(value)
        raise KeyError(f"No key containing {words} in {list(costs)}")
    refund = find("refund")
    churn  = find("churn")
    margin = find("margin", "profit")
    return refund, churn, margin


def cost_per_late_order(costs=COSTS):
    # Total cost of one late order = refund + (lost future orders x profit margin per order).
    refund, churn, margin = unpack_costs(costs)
    return refund + churn * margin


refund, churn, margin = unpack_costs(COSTS)
print(f"Refund per late order:      ${refund:.2f}")
print(f"Future orders lost (churn): {churn}")
print(f"Profit margin per order:    ${margin:.2f}")
print(f"Total cost per late order:  ${cost_per_late_order(COSTS):.2f}")
print(f"-> One late order wipes out the profit of about {cost_per_late_order(COSTS) / margin:.1f} on-time orders.")

## II(b) Choosing the best promise

### Net profit for a given promise
For a zone, a time block and a promise *P*:

$$\text{Net profit}(P) = N(P)\times\text{margin} \;-\; L(P)\times\text{cost per late order}$$

where $N(P)$ is the number of orders, i.e. `len(delivery_times(...))`, and $L(P)$ is the number of those orders delivered after *P*.

This captures the trade-off Rosa faces:
- A **shorter promise** attracts more orders, so $N$ rises. But more orders make deliveries slower, and the bar is tighter, so $L$ rises sharply.
- A **longer promise** makes lateness rare. But it drives customers away, so $N$ falls.

In [ ]:
def net_profit(zone, time_block, promise, costs=COSTS, seed=SEED):
    # Net profit over the four simulated weeks for one zone / time block / promise.
    refund, churn, margin = unpack_costs(costs)
    times = delivery_times(zone, time_block, promise, seed=seed)
    n_orders = len(times)
    n_late = int(np.sum(times > promise))
    return n_orders * margin - n_late * (refund + churn * margin)

### How to choose the range of promises properly
Before trying promises, we look at the actual delivery-time distribution in each segment, and at how profit behaves over a deliberately wide range.

In [ ]:
print(f"{'Zone':<12}{'Time block':<15}{'Min':>6}{'P10':>6}{'P50':>6}{'P90':>6}{'Max':>6}")
for z in ZONES:
    for b in TIME_BLOCKS:
        t = delivery_times(z, b, 45, seed=SEED)
        p = np.percentile(t, [10, 50, 90])
        print(f"{z:<12}{b:<15}{t.min():>6.1f}{p[0]:>6.1f}{p[1]:>6.1f}{p[2]:>6.1f}{t.max():>6.1f}")

In [ ]:
# Net profit over a deliberately wide scan (5 to 120 minutes) for one segment.
wide = list(range(5, 121, 5))
zone_ex, block_ex = "Far West", "Fri/Sat eve"
curve = [net_profit(zone_ex, block_ex, p, COSTS) for p in wide]

plt.figure(figsize=(8, 4))
plt.plot(wide, curve, marker="o")
plt.axvline(PROMISE, linestyle="--", color="grey", label="Current promise (45)")
plt.xlabel("Promised delivery time (minutes)")
plt.ylabel("Net profit over 4 weeks ($)")
plt.title(f"Net profit vs promise: {zone_ex}, {block_ex}")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Discussion: choosing the range.** The range should be wide enough to contain the true optimum, but should not waste effort on promises that cannot possibly win.

- **Lower bound.** A promise shorter than almost every delivery (well below the minimum and P10 in the table above) makes nearly every order late. Each late order loses money, because the cost per late order is several times the margin. Net profit there is deeply negative. The range example in the brief starts at 5 minutes, which is not realistic: no pizza is cooked and delivered in 5 minutes. A sensible floor sits a little below the fastest realistic deliveries, around the lowest P10 across segments.
- **Upper bound.** Once the promise is beyond almost all delivery times (around the maximum or P99), the late rate is essentially zero. Any further lengthening only loses customers, so profit can only fall. There is no point searching beyond that. The range example in the brief, which stops at 50 minutes, is too short for this reason. In slow segments such as the Far West on weekend evenings, 50 minutes may still sit in the tail, and the best promise could be longer.
- **Step size.** Steps of 5 minutes give a quick, readable scan and match how a promise would be communicated to customers ("45 minutes", "50 minutes"). If needed, a second pass in 1-minute steps around the winner can refine it.
- **Safeguard.** If the best promise falls **on the edge** of the range, the true optimum may lie outside it. The range should then be extended and the search re-run. The function below flags this automatically.

Based on the distribution table and the curve, we use **30 to 90 minutes in steps of 5** as the default range. It starts below the fast end of every segment and ends past the point where the late rate is essentially zero everywhere. The edge check below confirms this range is wide enough for every segment.

In [ ]:
def best_promise(zone, time_block, promises, costs=COSTS, seed=SEED):
    # Try every promise in `promises` and return the one with the highest net profit.
    #
    # Returns a dict with:
    #   best_promise, best_profit : the recommendation
    #   results                   : list of (promise, net profit) for every promise tried
    #   at_edge                   : True if the best promise is the shortest or longest one tried,
    #                               meaning the range should be widened
    promises = sorted(set(promises))
    results = []
    for p in promises:
        try:
            results.append((p, net_profit(zone, time_block, p, costs, seed)))
        except ValueError:
            continue  # skip any promise the simulator rejects
    if not results:
        raise ValueError("None of the promises could be evaluated.")

    # max() keeps the FIRST maximum, so ties go to the shorter (more attractive) promise
    best_p, best_np = max(results, key=lambda r: r[1])
    tried = [p for p, _ in results]
    return {
        "best_promise": best_p,
        "best_profit": best_np,
        "results": results,
        "at_edge": best_p in (tried[0], tried[-1]),
    }


PROMISE_RANGE = list(range(30, 91, 5))

out = best_promise("Far West", "Fri/Sat eve", PROMISE_RANGE, COSTS)
print("Far West, Fri/Sat eve")
for p, v in out["results"]:
    flag = "  <-- best" if p == out["best_promise"] else ""
    print(f"  promise {p:>3} min: net profit ${v:>10,.2f}{flag}")
print(f"\nRecommended promise: {out['best_promise']} minutes, net profit ${out['best_profit']:,.2f}")
print(f"Net profit at today's 45-minute promise: ${net_profit('Far West', 'Fri/Sat eve', 45, COSTS):,.2f}")
if out["at_edge"]:
    print("WARNING: best promise is at the edge of the range - widen the range and re-run.")

### Best promise for every zone and time block
We apply the same function to all segments and compare each result with today's single 45-minute promise.

In [ ]:
print(f"{'Zone':<12}{'Time block':<15}{'Best':>6}{'Net profit':>13}{'@45 min':>12}{'Gain':>11}{'Late %':>8}  Edge?")
total_best = total_45 = 0
summary = []
for z in ZONES:
    for b in TIME_BLOCKS:
        r = best_promise(z, b, PROMISE_RANGE, COSTS)
        p45 = net_profit(z, b, 45, COSTS)
        lr = late_rate(z, b, r["best_promise"])
        total_best += r["best_profit"]
        total_45 += p45
        summary.append((z, b, r["best_promise"]))
        print(f"{z:<12}{b:<15}{r['best_promise']:>6}{r['best_profit']:>13,.0f}{p45:>12,.0f}"
              f"{r['best_profit'] - p45:>11,.0f}{lr:>7.1f}%  {'YES - widen' if r['at_edge'] else 'no'}")

print(f"\nTotal net profit, today's 45-min promise everywhere: ${total_45:,.0f}")
print(f"Total net profit, best promise per segment:          ${total_best:,.0f}")
print(f"Improvement over four weeks:                         ${total_best - total_45:,.0f}")

In [ ]:
# Optional refinement: 1-minute steps within +/-4 minutes of the 5-minute winner (Far West, Fri/Sat eve)
coarse = best_promise("Far West", "Fri/Sat eve", PROMISE_RANGE, COSTS)["best_promise"]
fine = best_promise("Far West", "Fri/Sat eve", range(coarse - 4, coarse + 5), COSTS)
print(f"5-minute grid best: {coarse} min | 1-minute refinement best: {fine['best_promise']} min, "
      f"net profit ${fine['best_profit']:,.2f}")

### Recommendation
- **Stop using one promise everywhere.** A single 45-minute promise is too tight for slow segments, where it generates costly refunds and churn. It is also likely too loose for fast segments, where a shorter promise would win more orders at little risk. The table above gives the profit-maximising promise for each zone and time block, and the total shows the gain over today's policy.
- **Where the best promise is longer than 45 minutes** (expected mainly in the Far West and on weekend evenings, where late orders are concentrated), Rosa should promise more time. She gives up a few orders but avoids refunds and churn that cost several times an order's margin.
- **Where the best promise is shorter than 45 minutes**, Rosa can compete harder on speed.
- **Caveats.** The results come from a four-week simulation with one seed, so the exact values will move a little with different seeds. It is worth re-running with a few seeds before committing. The model holds driver staffing fixed and takes the cost figures in `COSTS` as given. The web app lets Rosa test how the recommendation changes if the margin, refund or churn estimates are different.

---
# Part III. Web app
The Streamlit app reuses `unpack_costs`, `net_profit` and `best_promise` from this notebook, moved into `rosa_logic.py`. It lets the user:
- select a zone and a time block from dropdown menus,
- set the range (and step) of promised times to try,
- adjust the profit margin per order, the churn per late order and the refund per late order,
- click **Find best promise** to see the recommendation, its net profit, the comparison with 45 minutes, and the full profit curve.

The notebook logic was carried into the app with GitHub Copilot, using the project skill `.github/skills/notebook-to-streamlit/SKILL.md`.

- Repository: `<paste GitHub repo URL>`
- Live app: `<paste Streamlit Community Cloud URL>`

---
# Appendix: Use of AI

**Tools used:** Claude (Anthropic) and GitHub Copilot in VS Code.

**How AI was used**
- **Claude** was used to draft the structure of the notebook and the first version of the functions for Parts I and II: the late-rate and average functions with `'all'` handling, the cost per late order, net profit, and the best-promise search. It also drafted the markdown explanations and the `notebook-to-streamlit` skill file. I reviewed the logic against the assignment brief and ran every cell in Colab with the real `rosa-starter` package. I checked the definitions (late = strictly after the promise; `'all'` pools orders rather than averaging percentages) and edited the written interpretation to match my own results.
- **GitHub Copilot** was used to build the Streamlit app. It was explicitly instructed to use the project skill `notebook-to-streamlit` so that the app reuses the notebook's logic instead of re-inventing it.
- I verified the app by checking that its recommendation for Far West / Fri/Sat eve with default costs matches the notebook output.

**Key prompts**
1. *(Claude)* "Complete as per the requirement" — with the assignment PDF attached.
2. *(Copilot)* "Use the notebook-to-streamlit skill. Using the logic in `rosa_assignment1.ipynb`, build a Streamlit app in `app.py` that lets Rosa select a zone and time block from dropdowns, set the range of promises to try, adjust the profit margin, churn per late order and refund per late order, and show the recommended promise after clicking a button."
3. *(Copilot)* *<add any follow-up prompts you used, e.g. fixing errors or deployment issues>*